# Cape Town neighbourhood clustering with geosnap-southafrica

Get the City of Cape Town's 116 wards, cluster them with geosnap and profile the result. First run downloads the ward boundaries and Stats SA workbook (a minute or two); later runs use the cache.

In [ ]:
from geosnap import DataStore
from geosnap.analyze import cluster
from geosnap_southafrica import get_south_africa

ct = get_south_africa(DataStore(), years=[2022], municipality='City of Cape Town')
ct[['geoid', 'ward_name', 'population', 'pct_black_african', 'pct_coloured', 'pct_white', 'youth_share', 'working_age_share']].head()

## Cluster the wards
k-means on population-group shares and age structure. `year` and `geoid` are the index columns geosnap expects.

In [ ]:
cols = ['pct_black_african', 'pct_coloured', 'pct_indian_asian', 'pct_white', 'youth_share', 'working_age_share', 'older_share']
clustered = cluster(ct, columns=cols, method='kmeans', n_clusters=5, temporal_index='year', unit_index='geoid')
clustered.explore('kmeans', categorical=True, tiles='CartoDB positron', legend=True)

## Profile the clusters

In [ ]:
clustered.groupby('kmeans').agg({**{c: 'mean' for c in cols}, 'population': 'sum'}).round(2)

## Optional: add environment and suburbs
These download from public servers on first use (see the README for sources and caveats).

In [ ]:
ct2 = get_south_africa(DataStore(), years=[2022], municipality='City of Cape Town',
                       extras=['landcover', 'elevation'], extras_errors='warn', suburbs='cape_town')
ct2[['ward_name', 'main_suburb', 'lc_share_tree', 'lc_share_built', 'elevation_mean']].head()

## Census 2011 to 2022 (needs the DataFirst files)
With the Community Profiles 2011 `.dta` files and the small-area polygons you can cluster on a two-census panel:

```python
ct = get_south_africa(years=[2011, 2022], municipality='City of Cape Town',
                      census2011='path/to/census2011', sal_geometry='path/to/sal_2011.shp')
```